In [2]:
import requests
import pandas as pd
import time
import os
import json

In [ ]:

# ── Config ────────────────────────────────────────────────────────────────────
APP_TOKEN  = "[your token!!!!]"
BASE_URL   = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
BATCH_SIZE = 50000
OUTPUT_DIR = "chicago_crime_data"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "progress.json")
FINAL_OUTPUT  = os.path.join(OUTPUT_DIR, "chicago_crimes_2001_2025.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Progress helpers ──────────────────────────────────────────────────────────
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {"offset": 0, "batch_num": 1, "total_records": 0}

def save_progress(offset, batch_num, total_records):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({
            "offset": offset,
            "batch_num": batch_num,
            "total_records": total_records
        }, f)

# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch_batch(offset, retries=3):
    params = {
        "$limit":  BATCH_SIZE,
        "$offset": offset,
        "$order":  "date ASC",
        "$$app_token": APP_TOKEN
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"  [Attempt {attempt}/{retries}] Request failed: {e}")
            if attempt < retries:
                time.sleep(5 * attempt)  # Exponential backoff
            else:
                raise

# ── Main download ─────────────────────────────────────────────────────────────
def download_all():
    progress  = load_progress()
    offset    = progress["offset"]
    batch_num = progress["batch_num"]
    total     = progress["total_records"]

    if offset > 0:
        print(f"Resuming from batch {batch_num} (offset={offset}, {total} records already saved)\n")
    else:
        print("Starting fresh download...\n")

    # Write CSV header only on first run
    write_header = not os.path.exists(FINAL_OUTPUT) or offset == 0
    file_mode = "w" if write_header else "a"

    with open(FINAL_OUTPUT, file_mode, encoding="utf-8") as csv_file:
        while True:
            print(f"Fetching batch {batch_num} (offset={offset})...")

            batch = fetch_batch(offset)

            if not batch:
                print("No more data. Download complete.")
                break

            batch_df = pd.DataFrame(batch)

            # Append to CSV (header only on first write)
            batch_df.to_csv(csv_file, index=False, header=write_header)
            write_header = False  # Only write header once

            total     += len(batch)
            offset    += BATCH_SIZE
            batch_num += 1

            save_progress(offset, batch_num, total)
            print(f"  -> {len(batch)} records fetched. Total so far: {total}")

            time.sleep(0.5)

    # Clean up progress file on successful completion
    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)
        print("Progress file removed.")

    print(f"\nDone! Total records: {total}")
    print(f"Saved to: {FINAL_OUTPUT}")

# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    download_all()

In [3]:
df = pd.read_csv("data/chicago_crime_data/chicago_crimes_2001_2025.csv", low_memory = False)

# Basic info
print(df.shape)          # rows and columns
print(df.dtypes)         # data types
print(df.columns.tolist()) # column names

(8500287, 22)
id                        int64
case_number                 str
date                        str
block                       str
iucr                        str
primary_type                str
description                 str
location_description        str
arrest                     bool
domestic                    str
beat                      int64
district                float64
ward                        str
community_area              str
fbi_code                    str
year                    float64
updated_on                  str
x_coordinate                str
y_coordinate                str
latitude                    str
longitude                   str
location                    str
dtype: object
['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'year', 'updated_on', 'x_coordinate', 'y_coordinate', 'latitude', 'longitude', 'locati

In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

print(missing_summary[missing_summary["missing_count"] > 0])

                      missing_count  missing_pct
longitude                    649205         7.64
location                     645357         7.59
latitude                      94655         1.11
year                          82006         0.96
updated_on                    79726         0.94
community_area                61165         0.72
ward                          49628         0.58
y_coordinate                  14929         0.18
location_description          15376         0.18
x_coordinate                  12649         0.15
fbi_code                      12702         0.15
district                         50         0.00


In [5]:
# Check the actual column order of a few rows from different batches
# Compare aligned vs misaligned rows side by side

aligned = df[df["year"].between(2001, 2025)].head(3)
misaligned_sample = df[~df["year"].between(2001, 2025)].head(3)

print("=== Aligned rows ===")
print(aligned[["year", "updated_on", "x_coordinate", "y_coordinate", "latitude", "longitude"]].to_string())

print("\n=== Misaligned rows ===")
print(misaligned_sample[["year", "updated_on", "x_coordinate", "y_coordinate", "latitude", "longitude"]].to_string())

# Also check: does the CSV header repeat inside the file? (another common cause)
# Read raw CSV to inspect
with open("data/chicago_crime_data/chicago_crimes_2001_2025.csv", "r") as f:
    lines = [f.readline() for _ in range(5)]
    
print("\n=== First 5 raw lines ===")
for i, line in enumerate(lines):
    print(f"Line {i}: {line[:200]}")

=== Aligned rows ===
     year               updated_on x_coordinate y_coordinate latitude longitude
0  2001.0  2023-10-21T15:42:03.000          NaN          NaN      NaN       NaN
1  2001.0  2025-01-11T15:40:58.000          NaN          NaN      NaN       NaN
2  2001.0  2025-01-11T15:40:58.000          NaN          NaN      NaN       NaN

=== Misaligned rows ===
             year updated_on x_coordinate             y_coordinate      latitude      longitude
650000  1170638.0    1831853         2002  2018-02-10T15:50:01.000  41.694080239  -87.650872151
650001  1158803.0    1862462         2002  2018-02-28T15:56:25.000  41.778325491  -87.693370087
650002  1156143.0    1862448         2002  2018-02-28T15:56:25.000  41.778340981  -87.703122242

=== First 5 raw lines ===
Line 0: id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,year,updated_on,x_coordinate,y_coordinate,latitude,longitude,loc
Line 1: 132466

In [6]:
!uv add pyarrow

df = pd.read_csv(
    "data/chicago_crime_data/chicago_crimes_2001_2025.csv",
    low_memory=False,
    dtype=str  # Read everything as string first, avoid dtype conflicts entirely
)

mask = df["year"].str.match(r"^\d{4}$") & df["year"].astype(float).between(2001, 2025)
mask_misaligned = ~mask

print(f"Aligned rows:    {mask.sum()}")
print(f"Misaligned rows: {mask_misaligned.sum()}")

# ── Fix misaligned rows by swapping columns ───────────────────────────────────
df.loc[mask_misaligned, ["year", "updated_on", "x_coordinate", "y_coordinate"]] = \
    df.loc[mask_misaligned, ["x_coordinate", "y_coordinate", "year", "updated_on"]].values

# ── Verify ────────────────────────────────────────────────────────────────────
print("\nYear distribution after fix:")
print(df["year"].value_counts().sort_index().head(30))

# ── Type conversion ───────────────────────────────────────────────────────────
df["date"]         = pd.to_datetime(df["date"], errors="coerce")
df["year"]         = pd.to_numeric(df["year"], errors="coerce")
df["x_coordinate"] = pd.to_numeric(df["x_coordinate"], errors="coerce")
df["y_coordinate"] = pd.to_numeric(df["y_coordinate"], errors="coerce")
df["latitude"]     = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"]    = pd.to_numeric(df["longitude"], errors="coerce")
df["arrest"]       = df["arrest"].map({"True": True, "False": False})
df["domestic"]     = df["domestic"].map({"True": True, "False": False})

# ── Filter valid years and save ───────────────────────────────────────────────
df = df[df["year"].between(2001, 2025)]
df["year"] = df["year"].astype(int)

print(f"\nFinal record count: {len(df)}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")

Resolved 116 packages in 17ms
Audited 113 packages in 26ms
Aligned rows:    750000
Misaligned rows: 7750287

Year distribution after fix:
year
2001                       485958
2002                       486831
2003                       475999
2004                       469443
2005                       453788
2006                       448198
2007                       437108
2008                       427216
2009                       392867
2010                       370566
2011                       352052
2012                       336374
2013                       307611
2014                       275880
2015                       264887
2016                       269957
2017                       269284
2018                       269146
2019                       261700
2020                       212697
2021                       182438
2021-11-20T15:41:20.000       266
2021-11-21T15:40:30.000       499
2021-11-22T15:40:20.000       441
2021-11-23T15:39:48.000        55
2021-11

In [8]:
# Filter out rows where year is a timestamp string (second-layer misalignment)
df = df[pd.to_numeric(df["year"], errors="coerce").notna()]
df["year"] = df["year"].astype(int)
df = df[df["year"].between(2001, 2025)]

print(f"Final clean record count: {len(df)}")
print(f"Year range: {df['year'].min()} - {df['year'].max()}")
print(f"\nYear distribution:")
print(df["year"].value_counts().sort_index())

Final clean record count: 8425935
Year range: 2001 - 2025

Year distribution:
year
2001    485958
2002    486831
2003    475999
2004    469443
2005    453788
2006    448198
2007    437108
2008    427216
2009    392867
2010    370566
2011    352052
2012    336374
2013    307611
2014    275880
2015    264887
2016    269957
2017    269284
2018    269146
2019    261700
2020    212697
2021    182438
2022    217208
2023    263273
2024    258988
2025    236466
Name: count, dtype: int64


In [ ]:
# Save
df.to_parquet("data/chicago_crime_data/chicago_crimes_cleaned.parquet", index=False)
print("Saved to parquet.")